Frequency Calculation Function

In [74]:
import sqlite3 as sql 
import numpy as np, pandas as pd

###connect to database and extract data - convert to dataframe

conn = sql.connect(r'..\data\KCM_data\kcmetro_data.db')
cursor = conn.cursor()

names = cursor.execute('SELECT name FROM sqlite_master WHERE type="table"').fetchall()
print('Table Names:\n',names)

data = pd.read_sql('SELECT * FROM kcm', conn)


Table Names:
 [('kcm',)]


In [111]:
def calc_route_freq(data, start_time, end_time, routes='all'):
    ###select data by criteria, give 5 minute buffer on each tail to account for closely times arrivals 
    start_time = pd.to_timedelta(start_time) - pd.Timedelta(minutes=5)
    end_time = pd.to_timedelta(end_time) + pd.Timedelta(minutes=5)

    data.departure_time = pd.to_timedelta(data.departure_time)
    ##select routes if necessary 
    if routes != 'all':
        data = data[data['route_short_name'].isin(routes)]

    ###select data by time range
    data = data[data['departure_time'].between(start_time, end_time)]

    ###soret by trip id, then direction id, then departure time
    data = data.sort_values(by=['route_id','direction_id', 'stop_id', 'departure_time'])

    return data


calc_route_freq(data, '07:00:00', '09:00:00', routes=['1','2','3']).groupby(['route_short_name', 'service_id', 'stop_id', 'direction_id']).departure_time.count()




route_short_name  service_id  stop_id  direction_id
1                 10087       538      0               4
                              558      0               4
                              575      0               4
                              600      0               4
                              605      0               4
                                                      ..
3                 54336       13441    0               3
                              13460    1               4
                              13480    1               4
                              13500    1               4
                              13510    1               4
Name: departure_time, Length: 612, dtype: int64

In [76]:
####commit and close connection
conn.commit()
conn.close()